# 02 — Modelagem Relacional e Normalização

**Objetivos:** projetar chaves, relações, dependências funcionais e evitar anomalias de atualização

**Como estudar:** leia a explicação, execute os exemplos, interprete os resultados, refaça os exercícios e consulte o resumo. Exemplos sintéticos são identificados; serviços externos são opcionais e precisam estar disponíveis para laboratórios de infraestrutura.

### Materiais complementares

- [Guia técnico — Modelagem relacional e normalização](./reference_materials/02_modelagem_relacional_normalizacao.pdf)
- [Cheat sheet — Modelagem relacional e normalização](./cheat_sheets/cheat_sheet_modelagem_relacional_normalizacao.pdf)
- [Documentação e arquivos de resultados](./README.pt-BR.md)


In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.cwd()
DATA = BASE / 'datasets'
OUT = BASE / 'outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
assert (DATA / 'leituras_sensores.csv').exists(), 'Abra o JupyterLab na raiz da disciplina (onde está datasets/).'


## Entidades e cardinalidade

No caso IoT, cômodos possuem vários sensores e sensores geram várias leituras (relações 1:N). A chave estrangeira de sensores aponta para cômodos; leituras aponta para sensores.

In [2]:
import sqlite3
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys=ON')
conn.executescript((Path('sql')/'01_schema_sqlite.sql').read_text(encoding='utf8'))
leituras = pd.read_csv(DATA/'leituras_sensores.csv')
leituras.to_sql('leituras',conn,if_exists='append',index=False)
print('SQLite inicializado:',conn.execute('SELECT COUNT(*) FROM leituras').fetchone()[0], 'leituras')


SQLite inicializado: 72 leituras


## Normalização de tabelas

**1FN:** valores atômicos e sem listas em colunas. **2FN:** sem dependência parcial de chave composta. **3FN:** sem dependência transitiva entre atributos não chave. Normalização previne anomalias; desnormalização deve responder a uma necessidade medida.

In [3]:
q='''SELECT c.nome AS comodo,s.id_sensor,l.id_evento,l.temperatura_c FROM comodos c JOIN sensores s ON c.id_comodo=s.id_comodo JOIN leituras l ON s.id_sensor=l.id_sensor LIMIT 8''';display(pd.read_sql_query(q,conn))

,comodo,id_sensor,id_evento,temperatura_c
0,Quarto,2,1,22.62
1,Cozinha,3,2,23.23
2,Sala,1,3,21.21
3,Quarto,2,4,23.38
4,Cozinha,3,5,23.67
5,Sala,1,6,19.66
6,Quarto,2,7,20.78
7,Cozinha,3,8,24.35


## Integridade referencial — experimento

Tentaremos inserir uma leitura vinculada a sensor inexistente. Em um esquema correto, a operação falha e não grava a linha.

In [4]:
try:
 conn.execute("INSERT INTO leituras VALUES (9999,999,'2025-01-01',21,50)")
except sqlite3.IntegrityError as exc:
 print('Restrição funcionou:',exc)
else:
 raise AssertionError('A inserção inválida não deveria ter sido permitida')

Restrição funcionou: FOREIGN KEY constraint failed


## Exercício resolvido — identificar anomalia

Se o nome do cômodo aparecesse repetido em toda leitura, sua mudança exigiria atualizar várias linhas. Separar `comodos` evita inconsistência. **Verificação:** a tabela de leituras guarda somente `id_sensor`.

In [5]:
print('Colunas:',[r[1] for r in conn.execute('PRAGMA table_info(leituras)')])

Colunas: ['id_evento', 'id_sensor', 'data_hora', 'temperatura_c', 'umidade_pct']


## Consulta Rápida e critérios de revisão

- **Antes de executar:** confirme o diretório de trabalho e as dependências do notebook.
- **Antes de concluir:** separe fatos observados, hipóteses, limitações e resultados sintéticos.
- **Antes de publicar:** não inclua credenciais, dados pessoais, volumes de bancos ou PDFs privados.
- **Exercício adicional:** altere uma hipótese do exemplo e explique o que mudou na saída, com evidência do código.

---
### Aprofundamento v2 — Dependências funcionais e anomalias

Uma dependência funcional `A → B` afirma que o mesmo valor de `A` determina `B` no domínio da relação; amostras finitas não comprovam uma regra de negócio. Revise **1FN**, **2FN**, **3FN** e **BCNF** segundo a semântica das dependências, distinguindo desnormalização deliberada de modelagem acidental.

**Interpretação:** compare o resultado com a teoria anterior; este exemplo local não substitui um laboratório de serviço externo.

In [6]:
import sqlite3
with sqlite3.connect(':memory:') as db:
    db.execute('PRAGMA foreign_keys=ON')
    db.execute('CREATE TABLE clientes(id INTEGER PRIMARY KEY, email TEXT NOT NULL UNIQUE)')
    db.execute('CREATE TABLE compras(id INTEGER PRIMARY KEY, cliente_id INTEGER REFERENCES clientes(id))')
    db.execute("INSERT INTO clientes VALUES (1, 'ana@exemplo.local')")
    db.execute('INSERT INTO compras VALUES (100, 1)')
    try:
        db.execute("INSERT INTO clientes VALUES (2, 'ana@exemplo.local')")
    except sqlite3.IntegrityError:
        print('Unicidade verificada')
    else:
        raise AssertionError('UNIQUE não aplicado')

Unicidade verificada


**Exercício resolvido — revisão:** identifique a principal hipótese do exemplo acima, explique por que ela importa e cite uma condição na qual a técnica falharia. **Resposta orientadora:** registre a pré-condição (tipo de dado, chave, restrição, volume ou topologia), teste-a explicitamente e explique os limites dos resultados observados.

### Resultado reproduzível em `outputs/`

A célula abaixo exporta uma evidência local do estudo. Não conecta a servidores externos.

In [7]:
pd.read_sql_query('SELECT s.id_sensor, c.nome AS comodo FROM sensores s JOIN comodos c ON c.id_comodo=s.id_comodo ORDER BY s.id_sensor', conn).to_csv(OUT / '02_sensores_comodos.csv',index=False)
print('Exportado: outputs/02_sensores_comodos.csv')

Exportado: outputs/02_sensores_comodos.csv
